# apatch — Туториал для начинающих

Этот ноутбук покажет вам самые базовые вещи:
- Как создать тестовый файл
- Как применить к нему патч через apatch
- Как сделать бэкап и откат

**Никаких моков. Всё реально.**

In [ ]:
import os, tempfile, shutil
from apatch.matcher import ASTMatcher
from apatch.backup import BackupManager

print('Все модули загружены!')

## Шаг 1: Создаём тестовый файл

In [ ]:
sandbox = os.path.join(tempfile.gettempdir(), 'apatch_beginner_sandbox')
if os.path.exists(sandbox):
    shutil.rmtree(sandbox)
os.makedirs(sandbox)

target_file = os.path.join(sandbox, 'hello.py')
with open(target_file, 'w') as f:
    f.write('def greet(name):\n'
            '    print(f"Hello, {name}!")\n'
            '    return True\n')

print(f'Файл создан: {target_file}')
print(open(target_file).read())

## Шаг 2: Точное совпадение (Level 1)

In [ ]:
matcher = ASTMatcher(target_file)

old_str = ('def greet(name):\n'
           '    print(f"Hello, {name}!")\n'
           '    return True')

new_str = ('def greet(name):\n'
           '    print(f"Hello, {name}! Welcome!")\n'
           '    log_greeting(name)\n'
           '    return True')

success, result, strategy = matcher.apply_patch(old_str, new_str)
print(f'Успех: {success}, Стратегия: {strategy}')

if success:
    with open(target_file, 'w') as f:
        f.write(result)
    print('Файл обновлён:')
    print(open(target_file).read())

## Шаг 3: Бэкап и откат

BackupManager сохраняет файл ДО изменений.

In [ ]:
# Свежий файл
with open(target_file, 'w') as f:
    f.write('def add(a, b):\n    return a + b\n')

print('ДО патча:')
print(open(target_file).read())

# Бэкап
bm = BackupManager(sandbox)
bm.create_backup(target_file)
print('Бэкап создан.')

# "Ломаем" файл
with open(target_file, 'w') as f:
    f.write('BROKEN CODE!!!')
print('ПОСЛЕ поломки:', open(target_file).read())

# Откат
bm.restore_file(target_file)
print('ПОСЛЕ отката:', open(target_file).read())
print('Файл восстановлен!')

In [ ]:
shutil.rmtree(sandbox)
print('Sandbox удалён.')